# X-OPT - TSPLIB FACILITY REMOVAL REOPTIMIZATION GROUP SUMMARIES


This notebook is a TSPLIB-oriented copy of the connected max-k-cut facility-removal experiment.

It uses the same instances grouped in `6.3.1 Geometric P-Median Interpretability - Connected Components.ipynb`:

- `WORK`
- `NOT WORK`
- `NOT NECESSARY`

For each source instance, the metaheuristic best solution is used as the reference solution. If the reference solution has `p` facilities, the notebook generates `p` reoptimization scenarios: one scenario for excluding each reference facility. In each scenario, the other `p-1` facilities are fixed and each variant chooses the candidate replacement pool. Because only one replacement facility is missing, each reoptimization is solved exactly by enumerating candidate replacements and evaluating the TSPLIB assignment cost.

The notebook returns:

1. raw results by source instance, removed facility, and variant;
2. a summary by variant across all reoptimization scenarios;
3. a summary by group and variant;
4. a wide group summary whose columns contain each variant's mean gap to baseline and mean runtime.


### SETUP


In [1]:
from __future__ import annotations

import gc
import os
import re
import sys
import math

from itertools import combinations
from pathlib   import Path
from time      import perf_counter

import networkx as nx
import numpy    as np
import pandas   as pd

from tqdm.auto       import tqdm
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)


In [2]:
# Make the sibling ./lib package importable when the kernel starts either
# from this notebook directory or from the project root.
for candidate in [Path.cwd(), Path.cwd() / "notebooks" / "experiments_sbpo", *Path.cwd().parents]:
    if (candidate / "lib" / "paths.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from lib.paths import find_project_root
from lib.graph import build_top_ltm
from lib.ltm import (
    binary_facility_vector,
    facilities_from_binary_vector,
    pmedian_solution_cost,
    build_solution_swap_graph,
    build_swap_component_graph,
    greedy_swap_path_between_solutions,
    complete_long_term_memory_with_swap_trajectories,
)
from lib.maxcut import best_facility_separation
from lib.tsplib import (
    TSPLIBInstanceSpec,
    assignment_profile,
    load_tsplib_instance,
    solve_geometric_instance,
    active_cooccurrence_matrix,
    max_k_cut_connected_partition,
)
from lib.utils import (
    parse_optional_int_env,
    parse_optional_float_env,
    as_sorted_tuple,
    format_facilities,
    format_groups,
)

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")


PROJECT_ROOT = /home/rei-luisinho/xopt


### CONFIGURATION


In [3]:
def first_optional_int_env(*names: str, default: int | None = None) -> int | None:
    for name in names:
        value = parse_optional_int_env(name)
        if value is not None:
            return value
    return default


def first_optional_float_env(*names: str, default: float | None = None) -> float | None:
    for name in names:
        value = parse_optional_float_env(name)
        if value is not None:
            return value
    return default


OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "experiments_sbpo" / "artifacts"

RAW_RESULTS_CSV           = OUTPUT_DIR / "facility_removal_tsplib_group_raw.csv"
STRUCTURES_CSV            = OUTPUT_DIR / "facility_removal_tsplib_group_structures.csv"
VARIANT_SUMMARY_CSV       = OUTPUT_DIR / "facility_removal_tsplib_group_variant_summary.csv"
GROUP_VARIANT_SUMMARY_CSV = OUTPUT_DIR / "facility_removal_tsplib_group_variant_by_group_summary.csv"
GROUP_SUMMARY_CSV         = OUTPUT_DIR / "facility_removal_tsplib_group_summary_wide.csv"
FAILURES_CSV              = OUTPUT_DIR / "facility_removal_tsplib_group_failures.csv"

TOP_FRACTION = first_optional_float_env("FACREM_TSPLIB_TOP_FRACTION", default=0.05)
GLOBAL_SEED  = first_optional_int_env  ("FACREM_TSPLIB_GLOBAL_SEED" , default=42)

SOLVER_PARAMS = {
    "restarts"       : first_optional_int_env("FACREM_TSPLIB_META_RESTARTS"    , default=10),
    "max_iter"       : first_optional_int_env("FACREM_TSPLIB_META_MAX_ITER"    , default=30),
    "swap_samples"   : first_optional_int_env("FACREM_TSPLIB_META_SWAP_SAMPLES", default=64),
    "seed"           : GLOBAL_SEED,
    "details_format" : "binary",
}

FACTOR = first_optional_float_env("FACREM_TSPLIB_META_FACTOR")
if FACTOR is not None:
    SOLVER_PARAMS["factor"] = FACTOR

MAX_K_CUT_RESTARTS = first_optional_int_env("FACREM_TSPLIB_MAX_K_CUT_RESTARTS", default=40)
MAX_K_CUT_MAX_ITER = first_optional_int_env("FACREM_TSPLIB_MAX_K_CUT_MAX_ITER", default=2000)

GROUP_FILTER     = os.getenv("FACREM_TSPLIB_GROUP_FILTER")
INSTANCE_FILTER  = os.getenv("FACREM_TSPLIB_INSTANCE_FILTER")
MAX_INSTANCES    = first_optional_int_env("FACREM_TSPLIB_MAX_INSTANCES")
SAVE_RESULTS_CSV = os.getenv("FACREM_TSPLIB_SAVE_RESULTS_CSV", "1") != "0"


In [4]:
INSTANCE_GROUPS = {
    "WORK": [
        {"name": "kroB200", "dimension": 200, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/kroB200.tsp.gz", "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/kroB200.tsp"},
        {"name": "lin105" , "dimension": 105, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/lin105.tsp.gz" , "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/lin105.tsp" },
        {"name": "rd100"  , "dimension": 100, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/rd100.tsp.gz"  , "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/rd100.tsp"  },
        {"name": "u574"   , "dimension": 574, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/u574.tsp.gz"   , "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/u574.tsp"   },
    ],
    "NOT WORK": [
        {"name": "pr136"  , "dimension": 136, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/pr136.tsp.gz"  , "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/pr136.tsp"  },
        {"name": "rat195" , "dimension": 195, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/rat195.tsp.gz" , "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/rat195.tsp" },
        {"name": "bier127", "dimension": 127, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/bier127.tsp.gz", "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/bier127.tsp"},
        {"name": "pr226"  , "dimension": 226, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/pr226.tsp.gz"  , "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/pr226.tsp"  },
    ],
    "NOT NECESSARY": [
        {"name": "a280"  , "dimension": 280, "edge_weight_type": "EUC_2D", "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/a280.tsp.gz"  , "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/a280.tsp"  },
        {"name": "ali535", "dimension": 535, "edge_weight_type": "GEO"   , "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/ali535.tsp.gz", "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/ali535.tsp"},
        {"name": "att48" , "dimension": 48 , "edge_weight_type": "ATT"   , "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/att48.tsp.gz" , "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/att48.tsp" },
        {"name": "att532", "dimension": 532, "edge_weight_type": "ATT"   , "p": 5, "url": "https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/att532.tsp.gz", "raw_path": "notebooks/experiments_sbpo/artifacts/tsplib_tsp/att532.tsp"},
    ],
}

GROUP_ORDER = ["WORK", "NOT WORK", "NOT NECESSARY"]


In [5]:
def spec_from_config(config: dict[str, object]) -> TSPLIBInstanceSpec:
    raw_path = Path(config["raw_path"])
    if not raw_path.is_absolute():
        raw_path = PROJECT_ROOT / raw_path

    return TSPLIBInstanceSpec(
        name=str(config["name"]),
        dimension=int(config["dimension"]),
        edge_weight_type=str(config["edge_weight_type"]).upper(),
        p=int(config["p"]),
        url=str(config["url"]),
        raw_path=raw_path,
    )


def selected_instance_rows() -> list[dict[str, object]]:
    rows = []
    allowed_groups = None
    if GROUP_FILTER:
        allowed_groups = {value.strip().upper() for value in GROUP_FILTER.split(",") if value.strip()}

    instance_regex = re.compile(INSTANCE_FILTER) if INSTANCE_FILTER else None

    for group in GROUP_ORDER:
        if allowed_groups is not None and group.upper() not in allowed_groups:
            continue
        for config in INSTANCE_GROUPS[group]:
            name = str(config["name"])
            if instance_regex is not None and not instance_regex.search(name):
                continue
            rows.append({"group": group, **config})

    if MAX_INSTANCES is not None:
        rows = rows[:MAX_INSTANCES]

    return rows


INSTANCE_ROWS = selected_instance_rows()

if not INSTANCE_ROWS:
    raise ValueError("No TSPLIB instances were selected.")

print(f"Selected source instances : {len(INSTANCE_ROWS)}")
print(f"Expected reopt scenarios  : {sum(int(row['p']) for row in INSTANCE_ROWS)}")
print(f"Groups                   : {', '.join(GROUP_ORDER)}")
print(f"Solver params            : {SOLVER_PARAMS}")
print(f"Top fraction             : {TOP_FRACTION:.0%}")
print(f"Max-k-cut params         : restarts={MAX_K_CUT_RESTARTS}, max_iter={MAX_K_CUT_MAX_ITER}, seed={GLOBAL_SEED}")

display(pd.DataFrame(INSTANCE_ROWS)[["group", "name", "dimension", "edge_weight_type", "p"]])


Selected source instances : 12
Expected reopt scenarios  : 60
Groups                   : WORK, NOT WORK, NOT NECESSARY
Solver params            : {'restarts': 10, 'max_iter': 30, 'swap_samples': 64, 'seed': 42, 'details_format': 'binary'}
Top fraction             : 5%
Max-k-cut params         : restarts=40, max_iter=2000, seed=42


,group,name,dimension,edge_weight_type,p
0,WORK,kroB200,200,EUC_2D,5
1,WORK,lin105,105,EUC_2D,5
2,WORK,rd100,100,EUC_2D,5
3,WORK,u574,574,EUC_2D,5
4,NOT WORK,pr136,136,EUC_2D,5
5,NOT WORK,rat195,195,EUC_2D,5
6,NOT WORK,bier127,127,EUC_2D,5
7,NOT WORK,pr226,226,EUC_2D,5
8,NOT NECESSARY,a280,280,EUC_2D,5
9,NOT NECESSARY,ali535,535,GEO,5


### MEMORY AND MAX-K-CUT HELPERS


In [6]:
def ltm_cost_range(long_term_memory: list[dict[str, object]]) -> tuple[float, float]:
    costs = [float(record["cost"]) for record in long_term_memory if record.get("cost") is not None]
    if not costs:
        return np.nan, np.nan
    return float(min(costs)), float(max(costs))


def ltm_records_to_solutions(long_term_memory: list[dict[str, object]]) -> list[tuple[int, ...]]:
    return [facilities_from_binary_vector(record["facilities"]) for record in long_term_memory]


def record_cost_or_compute(record: dict[str, object], solution: tuple[int, ...], cost_fn) -> float:
    cost = record.get("cost")
    if cost is None or pd.isna(cost):
        return float(cost_fn(solution))
    return float(cost)


def summarize_ltm_connectivity(long_term_memory: list[dict[str, object]]) -> dict[str, object]:
    if not long_term_memory:
        return {
            "solution_count": 0,
            "swap_edges": 0,
            "component_count": 0,
            "largest_component_size": 0,
            "smallest_component_size": 0,
            "is_connected": True,
        }

    solutions = ltm_records_to_solutions(long_term_memory)
    costs     = [float(record.get("cost", np.nan)) for record in long_term_memory]
    graph     = build_solution_swap_graph(solutions, costs)
    components = [sorted(component) for component in nx.connected_components(graph)]
    sizes      = [len(component) for component in components]

    return {
        "solution_count": len(solutions),
        "swap_edges": graph.number_of_edges(),
        "component_count": len(components),
        "largest_component_size": max(sizes) if sizes else 0,
        "smallest_component_size": min(sizes) if sizes else 0,
        "is_connected": graph.number_of_nodes() <= 1 or nx.is_connected(graph),
    }


def select_best_solution_component(
    long_term_memory: list[dict[str, object]],
    best_facilities: tuple[int, ...] | list[int],
) -> tuple[list[dict[str, object]], dict[str, object]]:
    if not long_term_memory:
        raise ValueError("long_term_memory is empty.")

    best_facilities = as_sorted_tuple(best_facilities)
    solutions = ltm_records_to_solutions(long_term_memory)
    costs = np.asarray([float(record.get("cost", np.nan)) for record in long_term_memory], dtype=float)

    try:
        best_index = solutions.index(best_facilities)
        reason = "exact_best_solution_match"
    except ValueError:
        best_index = int(np.nanargmin(costs))
        reason = "minimum_cost_record"

    graph = build_solution_swap_graph(solutions, costs.tolist())
    component_nodes = sorted(nx.node_connected_component(graph, best_index))
    selected_records = [dict(long_term_memory[index]) for index in component_nodes]

    return selected_records, {
        "best_component_solution_count": len(component_nodes),
        "best_component_selected_index": best_index,
        "best_component_selection_reason": reason,
    }


In [7]:
def complete_long_term_memory_pairwise_swap_trajectories(
    instance: object,
    long_term_memory: list[dict[str, object]],
    *,
    cost_fn=None,
) -> dict[str, object]:
    if not long_term_memory:
        raise ValueError("long_term_memory is empty.")

    n = int(np.asarray(long_term_memory[0]["facilities"], dtype=np.int8).size)
    effective_cost_fn = cost_fn if cost_fn is not None else lambda solution: pmedian_solution_cost(instance, solution)

    original_records = [dict(record) for record in long_term_memory]
    original_solutions = ltm_records_to_solutions(original_records)
    original_costs = [
        record_cost_or_compute(record, solution, effective_cost_fn)
        for record, solution in zip(original_records, original_solutions)
    ]

    for record, cost in zip(original_records, original_costs):
        vector = np.asarray(record["facilities"], dtype=np.int8)
        if vector.ndim != 1 or int(vector.size) != n:
            raise ValueError("all facility vectors must be one-dimensional and match instance size.")
        record["cost"] = float(cost)

    original_graph = build_solution_swap_graph(original_solutions, original_costs)
    original_is_connected = original_graph.number_of_nodes() <= 1 or nx.is_connected(original_graph)
    component_graph, components = build_swap_component_graph(original_graph, original_solutions)

    completed_records   = list(original_records)
    completed_solutions = list(original_solutions)
    solution_origin     = ["original"] * len(completed_solutions)
    solution_to_index   = {solution: index for index, solution in enumerate(completed_solutions)}
    trajectory_rows     = []

    if not original_is_connected:
        for source_component, target_component, edge_data in component_graph.edges(data=True):
            source_solution = original_solutions[int(edge_data["source_solution"])]
            target_solution = original_solutions[int(edge_data["target_solution"])]
            path, path_rows = greedy_swap_path_between_solutions(
                source_solution,
                target_solution,
                cost_fn=effective_cost_fn,
            )

            previous_solution = path[0]
            previous_index    = solution_to_index[previous_solution]

            for path_position, next_solution in enumerate(path[1:], start=1):
                if next_solution not in solution_to_index:
                    solution_to_index[next_solution] = len(completed_solutions)
                    completed_solutions.append(next_solution)
                    completed_records.append(
                        {
                            "cost": float(effective_cost_fn(next_solution)),
                            "facilities": binary_facility_vector(n, next_solution).tolist(),
                        }
                    )
                    solution_origin.append("trajectory")

                next_index = solution_to_index[next_solution]
                step_data  = path_rows[path_position - 1]
                trajectory_rows.append(
                    {
                        "source_component": int(source_component),
                        "target_component": int(target_component),
                        "from_solution": previous_index + 1,
                        "to_solution": next_index + 1,
                        "removed": step_data["removed"],
                        "added": step_data["added"],
                        "cost": step_data["cost"],
                        "swap_distance_to_goal": step_data["swap_distance_left"],
                        "solution": step_data["solution"],
                    }
                )
                previous_solution = next_solution
                previous_index    = next_index

    completed_costs = [
        record_cost_or_compute(record, solution, effective_cost_fn)
        for record, solution in zip(completed_records, completed_solutions)
    ]
    completed_graph = build_solution_swap_graph(completed_solutions, completed_costs)
    completed_is_connected = completed_graph.number_of_nodes() <= 1 or nx.is_connected(completed_graph)

    return {
        "long_term_memory": completed_records,
        "solutions": completed_solutions,
        "solution_origin": solution_origin,
        "trajectory_rows": trajectory_rows,
        "original_swap_graph": original_graph,
        "completed_swap_graph": completed_graph,
        "component_graph": component_graph,
        "components": components,
        "original_is_connected": original_is_connected,
        "completed_is_connected": completed_is_connected,
        "original_component_count": len(components),
        "completed_component_count": nx.number_connected_components(completed_graph),
        "original_solution_count": len(original_records),
        "completed_solution_count": len(completed_records),
        "added_solution_count": len(completed_records) - len(original_records),
        "movement_count": len(trajectory_rows),
        "component_edge_count": component_graph.number_of_edges(),
    }


def complete_long_term_memory_mst_with_range_guard(
    instance: object,
    long_term_memory: list[dict[str, object]],
    *,
    full_cost_range: tuple[float, float],
    best_facilities: tuple[int, ...] | list[int],
    cost_fn=None,
) -> dict[str, object]:
    result = complete_long_term_memory_with_swap_trajectories(
        instance,
        long_term_memory,
        cost_fn=cost_fn,
    )

    lower, upper = full_cost_range
    out_of_range = []
    for index, (record, origin) in enumerate(zip(result["long_term_memory"], result["solution_origin"])):
        if origin != "trajectory":
            continue
        cost = float(record["cost"])
        if cost < lower - 1e-9 or cost > upper + 1e-9:
            out_of_range.append({"record_index": index, "cost": cost})

    result["guard_fallback_used"] = False
    result["trajectory_generated_out_of_range_count"] = len(out_of_range)
    result["trajectory_generated_out_of_range_costs"] = out_of_range
    result["best_component_solution_count"] = np.nan
    result["best_component_selection_reason"] = None

    if not out_of_range:
        return result

    selected_records, metadata = select_best_solution_component(long_term_memory, best_facilities)
    selected_solutions = ltm_records_to_solutions(selected_records)
    selected_costs     = [float(record["cost"]) for record in selected_records]
    selected_graph     = build_solution_swap_graph(selected_solutions, selected_costs)

    result.update(
        {
            "long_term_memory": selected_records,
            "solutions": selected_solutions,
            "solution_origin": ["best_component"] * len(selected_records),
            "trajectory_rows": [],
            "completed_swap_graph": selected_graph,
            "completed_is_connected": selected_graph.number_of_nodes() <= 1 or nx.is_connected(selected_graph),
            "completed_component_count": nx.number_connected_components(selected_graph),
            "completed_solution_count": len(selected_records),
            "added_solution_count": 0,
            "movement_count": 0,
            "guard_fallback_used": True,
            **metadata,
        }
    )
    return result


In [8]:
def completion_edge_count(completion: dict[str, object] | None) -> float:
    if completion is None:
        return np.nan
    if "component_edge_count" in completion:
        return float(completion["component_edge_count"])
    component_mst = completion.get("component_mst")
    if component_mst is not None:
        return float(component_mst.number_of_edges())
    component_graph = completion.get("component_graph")
    if component_graph is not None:
        return float(component_graph.number_of_edges())
    return np.nan


def extract_max_k_cut_from_memory(
    instance,
    best_facilities: tuple[int, ...],
    long_term_memory: list[dict[str, object]],
    *,
    memory_strategy: str,
    completion: dict[str, object] | None = None,
) -> dict[str, object]:
    if not long_term_memory:
        raise ValueError("long_term_memory is empty.")

    matrix = np.vstack([np.asarray(record["facilities"], dtype=np.int8) for record in long_term_memory])
    costs  = np.asarray([float(record["cost"]) for record in long_term_memory], dtype=float)
    active_nodes, adjacency = active_cooccurrence_matrix(matrix)

    labels, groups, cut_weight, internal_weight, ignored_nodes = max_k_cut_connected_partition(
        active_nodes,
        adjacency,
        instance.p,
        len(instance.points),
        max_cut_restarts=MAX_K_CUT_RESTARTS,
        max_cut_max_iter=MAX_K_CUT_MAX_ITER,
        seed=GLOBAL_SEED,
    )

    connectivity = summarize_ltm_connectivity(long_term_memory)
    cost_min, cost_max = ltm_cost_range(long_term_memory)

    result = {
        "memory_strategy": memory_strategy,
        "analysis_memory_size": len(long_term_memory),
        "analysis_memory_min_cost": cost_min,
        "analysis_memory_max_cost": cost_max,
        "analysis_solution_count": connectivity["solution_count"],
        "analysis_swap_edges": connectivity["swap_edges"],
        "analysis_component_count": connectivity["component_count"],
        "analysis_largest_component_size": connectivity["largest_component_size"],
        "analysis_smallest_component_size": connectivity["smallest_component_size"],
        "analysis_is_connected": connectivity["is_connected"],
        "cooccurrence_active_nodes": int(active_nodes.size),
        "cooccurrence_edges": int(np.count_nonzero(np.triu(adjacency, 1) > 0)),
        "cooccurrence_total_weight": float(np.triu(adjacency, 1).sum()),
        "max_k_cut_groups": groups,
        "max_k_cut_ignored_nodes": len(ignored_nodes),
        "max_k_cut_fraction_cut": cut_weight / max(1e-12, cut_weight + internal_weight),
        "max_k_cut_best_facility_separation": best_facility_separation(labels, set(best_facilities)),
        "pre_connection_component_count": np.nan,
        "post_connection_component_count": connectivity["component_count"],
        "pre_connection_solution_count": np.nan,
        "post_connection_solution_count": len(long_term_memory),
        "trajectory_added_solution_count": 0,
        "trajectory_movement_count": 0,
        "trajectory_connection_edge_count": np.nan,
        "trajectory_generated_out_of_range_count": 0,
        "guard_fallback_used": False,
        "best_component_solution_count": np.nan,
        "best_component_selection_reason": None,
    }

    if completion is not None:
        result.update(
            {
                "pre_connection_component_count": completion.get("original_component_count"),
                "post_connection_component_count": completion.get("completed_component_count"),
                "pre_connection_solution_count": completion.get("original_solution_count"),
                "post_connection_solution_count": completion.get("completed_solution_count"),
                "trajectory_added_solution_count": completion.get("added_solution_count", 0),
                "trajectory_movement_count": completion.get("movement_count", 0),
                "trajectory_connection_edge_count": completion_edge_count(completion),
                "trajectory_generated_out_of_range_count": completion.get("trajectory_generated_out_of_range_count", 0),
                "guard_fallback_used": bool(completion.get("guard_fallback_used", False)),
                "best_component_solution_count": completion.get("best_component_solution_count", np.nan),
                "best_component_selection_reason": completion.get("best_component_selection_reason"),
            }
        )

    return result


def build_max_k_cut_memory_analyses(instance, result):
    full_ltm = result["details"]["long_term_memory"]
    if not full_ltm:
        raise ValueError("long_term_memory is empty.")

    best_facilities = as_sorted_tuple(result["selected"])
    top_ltm, _, top_costs = build_top_ltm(full_ltm, TOP_FRACTION)
    full_min, full_max = ltm_cost_range(full_ltm)
    top_min, top_max   = ltm_cost_range(top_ltm)
    cost_fn = lambda solution: assignment_profile(instance, solution)[0]

    base_metadata = {
        "source_ltm_size": len(full_ltm),
        "top_fraction": TOP_FRACTION,
        "top_solution_count": len(top_ltm),
        "top_cost_cutoff": float(top_costs.max()),
        "full_ltm_cost_min": full_min,
        "full_ltm_cost_max": full_max,
        "top_ltm_cost_min": top_min,
        "top_ltm_cost_max": top_max,
    }

    analyses = {}
    analyses["max_k_cut_original"] = extract_max_k_cut_from_memory(
        instance,
        best_facilities,
        top_ltm,
        memory_strategy="top_ltm_original",
    )

    mst_completion = complete_long_term_memory_with_swap_trajectories(
        instance,
        top_ltm,
        cost_fn=cost_fn,
    )
    analyses["max_k_cut_mst_trajectory"] = extract_max_k_cut_from_memory(
        instance,
        best_facilities,
        mst_completion["long_term_memory"],
        memory_strategy="top_ltm_mst_trajectory",
        completion=mst_completion,
    )

    pairwise_completion = complete_long_term_memory_pairwise_swap_trajectories(
        instance,
        top_ltm,
        cost_fn=cost_fn,
    )
    analyses["max_k_cut_pairwise_trajectory"] = extract_max_k_cut_from_memory(
        instance,
        best_facilities,
        pairwise_completion["long_term_memory"],
        memory_strategy="top_ltm_pairwise_trajectory",
        completion=pairwise_completion,
    )

    guarded_completion = complete_long_term_memory_mst_with_range_guard(
        instance,
        top_ltm,
        full_cost_range=(full_min, full_max),
        best_facilities=best_facilities,
        cost_fn=cost_fn,
    )
    analyses["max_k_cut_mst_range_guard"] = extract_max_k_cut_from_memory(
        instance,
        best_facilities,
        guarded_completion["long_term_memory"],
        memory_strategy="top_ltm_mst_trajectory_range_guard",
        completion=guarded_completion,
    )

    for analysis in analyses.values():
        analysis.update(base_metadata)

    return analyses, base_metadata


### REOPTIMIZATION HELPERS


In [9]:
def choose_removed_facility(open_facilities: tuple[int, ...] | list[int], position: int) -> int:
    facilities = as_sorted_tuple(open_facilities)
    if not facilities:
        raise ValueError("No facilities are available to remove.")
    if position < 0 or position >= len(facilities):
        raise ValueError(f"REMOVED_FACILITY_POSITION={position} is outside the solution size {len(facilities)}.")
    return facilities[position]


def find_group_containing(groups: list[tuple[int, ...]], facility: int) -> tuple[int, ...]:
    for group in groups:
        if facility in group:
            return tuple(sorted(group))
    raise ValueError(f"Facility {facility} was not found in any max-k-cut group.")


def build_max_k_cut_local_replacement_pool(groups, removed_facility):
    try:
        removed_group = find_group_containing(groups, removed_facility)
    except ValueError as exc:
        return tuple(), tuple(), f"{exc} Max-k-cut local reoptimization will be marked infeasible."

    replacement_pool = tuple(value for value in removed_group if value != removed_facility)
    if not replacement_pool:
        return removed_group, replacement_pool, (
            "The max-k-cut group that contains the removed facility has no replacement candidate after forbidding the removed facility."
        )
    return removed_group, replacement_pool, None


def solve_replacement_variant(
    *,
    variant: str,
    variant_order: int,
    instance,
    p: int,
    allowed_facilities=None,
    fixed_open_facilities=None,
    forbidden_facilities=None,
) -> tuple[dict[str, object], tuple[int, ...]]:
    started_at = perf_counter()
    n = len(instance.points)
    fixed_set = set() if fixed_open_facilities is None else {int(value) for value in fixed_open_facilities}
    forbidden_set = set() if forbidden_facilities is None else {int(value) for value in forbidden_facilities}

    if not fixed_set.isdisjoint(forbidden_set):
        raise ValueError("A fixed-open facility cannot also be forbidden.")

    if allowed_facilities is None:
        replacement_candidates = set(range(n))
    else:
        replacement_candidates = {int(value) for value in allowed_facilities}

    replacement_candidates.difference_update(forbidden_set)
    replacement_candidates.difference_update(fixed_set)
    replacement_candidates = sorted(replacement_candidates)

    missing_count = int(p) - len(fixed_set)

    row = {
        "variant": variant,
        "variant_order": variant_order,
        "candidate_facility_count": len(replacement_candidates),
        "fixed_facility_count": len(fixed_set),
        "forbidden_facility_count": len(forbidden_set),
        "candidate_facility_fraction": len(replacement_candidates) / max(1, n),
        "fixed_facilities": format_facilities(fixed_set),
        "forbidden_facilities": format_facilities(forbidden_set),
        "objective_value": np.nan,
        "model_build_seconds": 0.0,
        "solve_seconds": np.nan,
        "total_variant_seconds": np.nan,
        "evaluated_replacements": 0,
        "open_facilities_count": 0,
        "open_facilities": "",
        "status": "ERROR",
        "error_message": None,
    }

    if missing_count < 0:
        row.update({"status": "INVALID_FIXED_SET", "error_message": f"Too many fixed facilities: {len(fixed_set)} > {p}."})
        return row, tuple()

    if len(replacement_candidates) < missing_count:
        row.update({"status": "INFEASIBLE_POOL", "error_message": f"Candidate replacement pool smaller than missing count: {len(replacement_candidates)} < {missing_count}."})
        row["solve_seconds"] = perf_counter() - started_at
        row["total_variant_seconds"] = row["solve_seconds"]
        return row, tuple()

    if missing_count > 2:
        combination_count = math.comb(len(replacement_candidates), missing_count)
        row.update({"status": "UNSUPPORTED_ENUMERATION", "error_message": f"Enumeration would require {combination_count} combinations."})
        row["solve_seconds"] = perf_counter() - started_at
        row["total_variant_seconds"] = row["solve_seconds"]
        return row, tuple()

    best_cost = np.inf
    best_open = tuple()
    evaluated = 0

    try:
        for combo in combinations(replacement_candidates, missing_count):
            open_facilities = as_sorted_tuple([*fixed_set, *combo])
            cost = float(assignment_profile(instance, open_facilities)[0])
            evaluated += 1

            if cost < best_cost - 1e-9 or (abs(cost - best_cost) <= 1e-9 and open_facilities < best_open):
                best_cost = cost
                best_open = open_facilities

        if missing_count == 0:
            best_open = as_sorted_tuple(fixed_set)
            best_cost = float(assignment_profile(instance, best_open)[0])
            evaluated = 1

        solve_seconds = perf_counter() - started_at
        row.update(
            {
                "objective_value": best_cost,
                "solve_seconds": solve_seconds,
                "total_variant_seconds": solve_seconds,
                "evaluated_replacements": evaluated,
                "open_facilities_count": len(best_open),
                "open_facilities": format_facilities(best_open),
                "status": "OPTIMAL",
                "error_message": None,
            }
        )
        return row, best_open
    except Exception as exc:
        solve_seconds = perf_counter() - started_at
        row.update({"status": "ERROR", "solve_seconds": solve_seconds, "total_variant_seconds": solve_seconds, "error_message": f"{type(exc).__name__}: {exc}"})
        return row, tuple()
    finally:
        gc.collect()


def describe_solution_change(reference_open, candidate_open):
    reference_set = set(as_sorted_tuple(reference_open))
    candidate_set = set(as_sorted_tuple(candidate_open))
    overlap = sorted(reference_set.intersection(candidate_set))
    dropped = sorted(reference_set - candidate_set)
    added   = sorted(candidate_set - reference_set)
    return {
        "overlap_with_reference_count": len(overlap),
        "overlap_with_reference_fraction": len(overlap) / max(1, len(reference_set)),
        "dropped_reference_facilities_count": len(dropped),
        "new_facilities_count": len(added),
        "dropped_reference_facilities": format_facilities(dropped),
        "new_facilities": format_facilities(added),
    }


### BENCHMARK EXECUTION


In [10]:
VARIANT_SPECS = [
    {"variant": "baseline_all_facilities",        "variant_order": 1, "analysis_key": None},
    {"variant": "max_k_cut_original",            "variant_order": 2, "analysis_key": "max_k_cut_original"},
    {"variant": "max_k_cut_mst_trajectory",      "variant_order": 3, "analysis_key": "max_k_cut_mst_trajectory"},
    {"variant": "max_k_cut_pairwise_trajectory", "variant_order": 4, "analysis_key": "max_k_cut_pairwise_trajectory"},
    {"variant": "max_k_cut_mst_range_guard",     "variant_order": 5, "analysis_key": "max_k_cut_mst_range_guard"},
]
VARIANT_ORDER = [spec["variant"] for spec in VARIANT_SPECS]


def reoptimization_instance_name(instance_name: str, removal_position: int, removed_facility: int | None = None) -> str:
    if removed_facility is None or pd.isna(removed_facility):
        return f"{instance_name}__remove_pos_{removal_position}"
    return f"{instance_name}__remove_pos_{removal_position}__facility_{int(removed_facility) + 1}"


def build_pipeline_error_rows(row_config, error_message):
    rows = []
    instance_name = row_config.get("name")
    p_value = int(row_config.get("p", 0) or 0)

    for removal_position in range(max(1, p_value)):
        reopt_name = reoptimization_instance_name(str(instance_name), removal_position)
        for spec in VARIANT_SPECS:
            rows.append(
                {
                    "instance_group": row_config.get("group"),
                    "instance": instance_name,
                    "reoptimization_instance": reopt_name,
                    "removal_position": removal_position,
                    "removed_facility": np.nan,
                    "removed_facility_zero_based": np.nan,
                    "n": row_config.get("dimension", np.nan),
                    "p": row_config.get("p", np.nan),
                    "variant": spec["variant"],
                    "variant_order": spec["variant_order"],
                    "status": "PIPELINE_ERROR",
                    "error_message": error_message,
                }
            )

    return rows


def analysis_for_structure_row(analyses):
    row = {}
    for key, analysis in analyses.items():
        prefix = f"{key}__"
        for field in [
            "memory_strategy",
            "analysis_memory_size",
            "analysis_component_count",
            "analysis_is_connected",
            "cooccurrence_active_nodes",
            "cooccurrence_edges",
            "cooccurrence_total_weight",
            "max_k_cut_fraction_cut",
            "max_k_cut_best_facility_separation",
            "pre_connection_component_count",
            "post_connection_component_count",
            "trajectory_added_solution_count",
            "trajectory_movement_count",
            "trajectory_connection_edge_count",
            "trajectory_generated_out_of_range_count",
            "guard_fallback_used",
            "best_component_solution_count",
        ]:
            row[prefix + field] = analysis.get(field, np.nan)
        row[prefix + "groups"] = format_groups(analysis.get("max_k_cut_groups", []))
    return row


def build_reoptimization_scenario_rows(
    *,
    row_config,
    spec,
    instance,
    result,
    memory_analyses,
    memory_metadata,
    metaheuristic_runtime_seconds,
    structure_runtime_seconds,
    reference_open,
    removal_position,
    removed_facility,
):
    instance_name = str(row_config["name"])
    remaining_reference = tuple(facility for facility in reference_open if facility != removed_facility)
    reopt_name = reoptimization_instance_name(instance_name, removal_position, removed_facility)

    common = {
        "instance_group": str(row_config["group"]),
        "instance": instance_name,
        "reoptimization_instance": reopt_name,
        "instance_id": instance.name,
        "n": int(spec.dimension),
        "p": int(spec.p),
        "edge_weight_type": spec.edge_weight_type,
        "raw_path": str(spec.raw_path),
        "reference_source": "metaheuristic_best_solution",
        "reference_cost": float(result["summary"]["tspmed_cost"]),
        "reference_facilities": format_facilities(reference_open),
        "metaheuristic_runtime_seconds": metaheuristic_runtime_seconds,
        "structure_extraction_runtime_seconds": structure_runtime_seconds,
        "pre_reoptimization_seconds": metaheuristic_runtime_seconds + structure_runtime_seconds,
        "removed_facility": removed_facility + 1,
        "removed_facility_zero_based": removed_facility,
        "removal_position": removal_position,
        "remaining_reference_facilities": format_facilities(remaining_reference),
        "remaining_reference_facilities_count": len(remaining_reference),
        **memory_metadata,
    }

    rows = []
    for spec_variant in VARIANT_SPECS:
        allowed_facilities = None
        variant_analysis = None
        removed_group = tuple()
        replacement_pool = tuple()
        group_error = None

        if spec_variant["analysis_key"] is not None:
            variant_analysis = memory_analyses[spec_variant["analysis_key"]]
            removed_group, replacement_pool, group_error = build_max_k_cut_local_replacement_pool(
                variant_analysis["max_k_cut_groups"],
                removed_facility,
            )
            allowed_facilities = replacement_pool

        variant_row, open_facilities = solve_replacement_variant(
            variant=spec_variant["variant"],
            variant_order=spec_variant["variant_order"],
            instance=instance,
            p=int(spec.p),
            allowed_facilities=allowed_facilities,
            fixed_open_facilities=remaining_reference,
            forbidden_facilities=(removed_facility,),
        )

        if group_error is not None:
            variant_row["error_message"] = group_error

        variant_row.update(common)
        if variant_analysis is None:
            variant_row.update({"memory_strategy": "all_facilities_baseline"})
        else:
            variant_row.update({key: value for key, value in variant_analysis.items() if key != "max_k_cut_groups"})
            variant_row.update(
                {
                    "max_k_cut_removed_group_found": int(bool(removed_group)),
                    "max_k_cut_removed_group_size": len(removed_group),
                    "max_k_cut_replacement_pool_size": len(replacement_pool),
                    "max_k_cut_removed_group": format_facilities(removed_group),
                    "max_k_cut_replacement_pool": format_facilities(replacement_pool),
                    "max_k_cut_groups_formatted": format_groups(variant_analysis["max_k_cut_groups"]),
                }
            )

        variant_row.update(describe_solution_change(reference_open, open_facilities))
        variant_row["total_pipeline_seconds"] = common["pre_reoptimization_seconds"] + (
            variant_row["total_variant_seconds"] if pd.notna(variant_row["total_variant_seconds"]) else 0.0
        )
        rows.append(variant_row)

    structure_row = {**common, "error_message": None, **analysis_for_structure_row(memory_analyses)}

    return rows, structure_row


def run_single_instance_analysis(row_config):
    instance_name = str(row_config["name"])
    started_at = perf_counter()

    try:
        spec = spec_from_config(row_config)
        instance = load_tsplib_instance(spec)

        solve_started_at = perf_counter()
        result = solve_geometric_instance(instance, SOLVER_PARAMS)
        metaheuristic_runtime_seconds = perf_counter() - solve_started_at

        structure_started_at = perf_counter()
        memory_analyses, memory_metadata = build_max_k_cut_memory_analyses(instance, result)
        structure_runtime_seconds = perf_counter() - structure_started_at

        reference_open = as_sorted_tuple(result["selected"])
        if len(reference_open) != int(spec.p):
            raise ValueError(f"Expected reference solution with p={spec.p} facilities, found {len(reference_open)}.")

        rows = []
        structure_rows = []

        for removal_position, removed_facility in enumerate(reference_open):
            scenario_rows, structure_row = build_reoptimization_scenario_rows(
                row_config=row_config,
                spec=spec,
                instance=instance,
                result=result,
                memory_analyses=memory_analyses,
                memory_metadata=memory_metadata,
                metaheuristic_runtime_seconds=metaheuristic_runtime_seconds,
                structure_runtime_seconds=structure_runtime_seconds,
                reference_open=reference_open,
                removal_position=removal_position,
                removed_facility=removed_facility,
            )
            rows.extend(scenario_rows)
            structure_rows.append(structure_row)

        return rows, structure_rows
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        return (
            build_pipeline_error_rows(row_config, error_message),
            [
                {
                    "instance_group": row_config.get("group"),
                    "instance": row_config.get("name"),
                    "reoptimization_instance": reoptimization_instance_name(str(row_config.get("name")), removal_position),
                    "removal_position": removal_position,
                    "removed_facility": np.nan,
                    "removed_facility_zero_based": np.nan,
                    "n": row_config.get("dimension", np.nan),
                    "p": row_config.get("p", np.nan),
                    "pipeline_seconds": perf_counter() - started_at,
                    "error_message": error_message,
                }
                for removal_position in range(max(1, int(row_config.get("p", 0) or 0)))
            ],
        )


def run_benchmark(instance_rows):
    rows = []
    structure_rows = []

    for row_config in tqdm(instance_rows, total=len(instance_rows), desc="TSPLIB source-instance benchmark", dynamic_ncols=True):
        instance_rows_result, instance_structure_rows = run_single_instance_analysis(row_config)
        rows.extend(instance_rows_result)
        structure_rows.extend(instance_structure_rows)

    results_df = pd.DataFrame(rows)
    structures_df = pd.DataFrame(structure_rows)

    if not results_df.empty:
        results_df["variant_order"] = pd.to_numeric(results_df["variant_order"], errors="coerce")
        results_df["instance_group"] = pd.Categorical(results_df["instance_group"], categories=GROUP_ORDER, ordered=True)
        results_df = results_df.sort_values(
            ["instance_group", "instance", "removal_position", "variant_order"],
            kind="stable",
        ).reset_index(drop=True)

    if not structures_df.empty and "instance_group" in structures_df.columns:
        structures_df["instance_group"] = pd.Categorical(structures_df["instance_group"], categories=GROUP_ORDER, ordered=True)
        structures_df = structures_df.sort_values(
            ["instance_group", "instance", "removal_position"],
            kind="stable",
        ).reset_index(drop=True)

    return results_df, structures_df


In [11]:
def add_baseline_reference(results_df: pd.DataFrame) -> pd.DataFrame:
    merged = results_df.copy()
    if merged.empty:
        return merged

    baseline_df = (
        merged.loc[
            merged["variant"] == "baseline_all_facilities",
            [
                "reoptimization_instance",
                "status",
                "objective_value",
                "total_variant_seconds",
                "total_pipeline_seconds",
            ],
        ]
        .rename(
            columns={
                "status": "baseline_status",
                "objective_value": "baseline_objective_value",
                "total_variant_seconds": "baseline_total_variant_seconds",
                "total_pipeline_seconds": "baseline_total_pipeline_seconds",
            }
        )
    )

    merged = merged.merge(baseline_df, on="reoptimization_instance", how="left")
    merged["gap_to_baseline_percent"] = np.where(
        merged["objective_value"].notna() & merged["baseline_objective_value"].notna(),
        100.0 * (merged["objective_value"] - merged["baseline_objective_value"]) / merged["baseline_objective_value"],
        np.nan,
    )
    merged["time_delta_to_baseline_seconds"] = merged["total_variant_seconds"] - merged["baseline_total_variant_seconds"]
    return merged


def build_variant_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    working = results_df.copy()
    return (
        working.groupby(["variant_order", "variant"], dropna=False)
        .agg(
            source_instances=("instance", "nunique"),
            reoptimization_instances=("reoptimization_instance", "nunique"),
            solved_reoptimization_instances=("objective_value", lambda s: int(s.notna().sum())),
            optimal_reoptimization_instances=("status", lambda s: int((s == "OPTIMAL").sum())),
            mean_gap_to_baseline_percent=("gap_to_baseline_percent", "mean"),
            median_gap_to_baseline_percent=("gap_to_baseline_percent", "median"),
            max_gap_to_baseline_percent=("gap_to_baseline_percent", "max"),
            mean_total_variant_seconds=("total_variant_seconds", "mean"),
            mean_total_pipeline_seconds=("total_pipeline_seconds", "mean"),
            mean_time_delta_to_baseline_seconds=("time_delta_to_baseline_seconds", "mean"),
            mean_candidate_facility_count=("candidate_facility_count", "mean"),
            mean_replacement_pool_size=("max_k_cut_replacement_pool_size", "mean"),
            mean_trajectory_added=("trajectory_added_solution_count", "mean"),
        )
        .reset_index()
        .sort_values("variant_order")
        .reset_index(drop=True)
    )


def build_group_variant_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    working = results_df.copy()
    return (
        working.groupby(["instance_group", "variant_order", "variant"], observed=True, dropna=False)
        .agg(
            source_instances=("instance", "nunique"),
            reoptimization_instances=("reoptimization_instance", "nunique"),
            solved_reoptimization_instances=("objective_value", lambda s: int(s.notna().sum())),
            optimal_reoptimization_instances=("status", lambda s: int((s == "OPTIMAL").sum())),
            mean_gap_to_baseline_percent=("gap_to_baseline_percent", "mean"),
            median_gap_to_baseline_percent=("gap_to_baseline_percent", "median"),
            max_gap_to_baseline_percent=("gap_to_baseline_percent", "max"),
            mean_total_variant_seconds=("total_variant_seconds", "mean"),
            mean_total_pipeline_seconds=("total_pipeline_seconds", "mean"),
            mean_time_delta_to_baseline_seconds=("time_delta_to_baseline_seconds", "mean"),
        )
        .reset_index()
        .sort_values(["instance_group", "variant_order"])
        .reset_index(drop=True)
    )


def build_group_summary_wide(group_variant_summary_df: pd.DataFrame) -> pd.DataFrame:
    if group_variant_summary_df.empty:
        return pd.DataFrame()

    metric_columns = [
        "mean_gap_to_baseline_percent",
        "mean_total_variant_seconds",
        "mean_total_pipeline_seconds",
        "mean_time_delta_to_baseline_seconds",
    ]

    wide = group_variant_summary_df.pivot(
        index="instance_group",
        columns="variant",
        values=metric_columns,
    )
    wide.columns = [f"{variant}__{metric}" for metric, variant in wide.columns]
    wide = wide.reset_index()

    count_df = (
        group_variant_summary_df.groupby("instance_group", observed=True)
        .agg(
            source_instances=("source_instances", "max"),
            reoptimization_instances=("reoptimization_instances", "max"),
        )
        .reset_index()
    )

    return count_df.merge(wide, on="instance_group", how="left")


### APPLY


In [12]:
raw_results_df, structures_df = run_benchmark(INSTANCE_ROWS)


TSPLIB source-instance benchmark:   0%|          | 0/12 [00:00<?, ?it/s]

In [13]:
results_df = add_baseline_reference(raw_results_df)

variant_summary_df       = build_variant_summary(results_df)
group_variant_summary_df = build_group_variant_summary(results_df)
group_summary_wide_df    = build_group_summary_wide(group_variant_summary_df)

error_series = results_df.get("error_message", pd.Series(index=results_df.index, dtype=object))
failures_df  = results_df[error_series.notna()].copy()


In [14]:
display(results_df.head())


,variant,variant_order,candidate_facility_count,fixed_facility_count,forbidden_facility_count,candidate_facility_fraction,fixed_facilities,forbidden_facilities,objective_value,model_build_seconds,solve_seconds,total_variant_seconds,evaluated_replacements,open_facilities_count,open_facilities,status,error_message,instance_group,instance,reoptimization_instance,instance_id,n,p,edge_weight_type,raw_path,reference_source,reference_cost,reference_facilities,metaheuristic_runtime_seconds,structure_extraction_runtime_seconds,pre_reoptimization_seconds,removed_facility,removed_facility_zero_based,removal_position,remaining_reference_facilities,remaining_reference_facilities_count,source_ltm_size,top_fraction,top_solution_count,top_cost_cutoff,full_ltm_cost_min,full_ltm_cost_max,top_ltm_cost_min,top_ltm_cost_max,memory_strategy,overlap_with_reference_count,overlap_with_reference_fraction,dropped_reference_facilities_count,new_facilities_count,dropped_reference_facilities,new_facilities,total_pipeline_seconds,analysis_memory_size,analysis_memory_min_cost,analysis_memory_max_cost,analysis_solution_count,analysis_swap_edges,analysis_component_count,analysis_largest_component_size,analysis_smallest_component_size,analysis_is_connected,cooccurrence_active_nodes,cooccurrence_edges,cooccurrence_total_weight,max_k_cut_ignored_nodes,max_k_cut_fraction_cut,max_k_cut_best_facility_separation,pre_connection_component_count,post_connection_component_count,pre_connection_solution_count,post_connection_solution_count,trajectory_added_solution_count,trajectory_movement_count,trajectory_connection_edge_count,trajectory_generated_out_of_range_count,guard_fallback_used,best_component_solution_count,best_component_selection_reason,max_k_cut_removed_group_found,max_k_cut_removed_group_size,max_k_cut_replacement_pool_size,max_k_cut_removed_group,max_k_cut_replacement_pool,max_k_cut_groups_formatted,baseline_status,baseline_objective_value,baseline_total_variant_seconds,baseline_total_pipeline_seconds,gap_to_baseline_percent,time_delta_to_baseline_seconds
0,baseline_all_facilities,1,195,4,1,0.975,41 68 83 95,28,98068.0,0.0,0.017827,0.017827,195,5,7 41 68 83 95,OPTIMAL,None,WORK,kroB200,kroB200__remove_pos_0__facility_29,tsplib_kroB200_p5,200,5,EUC_2D,/home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/tsplib_tsp/kroB200.tsp,metaheuristic_best_solution,97512.0,28 41 68 83 95,0.655835,0.085215,0.741049,29,28,0,41 68 83 95,4,732,0.05,37,98071.0,97512.0,101598.0,97512.0,98071.0,all_facilities_baseline,4,0.8,1,1,28,7,0.758876,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OPTIMAL,98068.0,0.017827,0.758876,0.0,0.000000
1,max_k_cut_original,2,2,4,1,0.010,41 68 83 95,28,98068.0,0.0,0.000641,0.000641,2,5,7 41 68 83 95,OPTIMAL,None,WORK,kroB200,kroB200__remove_pos_0__facility_29,tsplib_kroB200_p5,200,5,EUC_2D,/home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/tsplib_tsp/kroB200.tsp,metaheuristic_best_solution,97512.0,28 41 68 83 95,0.655835,0.085215,0.741049,29,28,0,41 68 83 95,4,732,0.05,37,98071.0,97512.0,101598.0,97512.0,98071.0,top_ltm_original,4,0.8,1,1,28,7,0.741690,37.0,97512.0,98071.0,37.0,89.0,2.0,35.0,2.0,False,20.0,89.0,370.0,180.0,1.0,1.0,NaN,2.0,NaN,37.0,0.0,0.0,NaN,0.0,False,NaN,None,1.0,3.0,2.0,7 28 51,7 51,6 57 83 97 185 | 95 118 136 161 | 7 28 51 | 29 68 74 133 149 | 41 119 124,OPTIMAL,98068.0,0.017827,0.758876,0.0,-0.017186
2,max_k_cut_mst_trajectory,3,2,4,1,0.010,41 68 83 95,28,98068.0,0.0,0.000427,0.000427,2,5,7 41 68 83 95,OPTIMAL,None,WORK,kroB200,kroB200__remove_pos_0__facility_29,tsplib_kroB200_p5,200,5,EUC_2D,/home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/tsplib_tsp/kroB200.tsp,metaheuristic_best_solution,97512.0,28 41 68 83 95,0.655835,0.085215,0.741049,29,28,0,41 68 83 95,4,732,0.05,37,98071.0,97512.0,101598.0,97512.0,98071.0,top_ltm_mst_trajectory,4,0.8,1,1,28,7,0.741476,38.0,97512.0,98656.0,38.0,92.0,1.0,38.0,38.0,True,20.0,90.

In [15]:
display(variant_summary_df.round(4))


,variant_order,variant,source_instances,reoptimization_instances,solved_reoptimization_instances,optimal_reoptimization_instances,mean_gap_to_baseline_percent,median_gap_to_baseline_percent,max_gap_to_baseline_percent,mean_total_variant_seconds,mean_total_pipeline_seconds,mean_time_delta_to_baseline_seconds,mean_candidate_facility_count,mean_replacement_pool_size,mean_trajectory_added
0,1,baseline_all_facilities,12,60,60,60,0.0000,0.0,0.0000,0.0323,3.6617,0.0000,249.8333,NaN,NaN
1,2,max_k_cut_original,12,60,51,51,0.1172,0.0,3.9916,0.0006,3.6299,-0.0317,3.0667,3.0667,0.0000
2,3,max_k_cut_mst_trajectory,12,60,51,51,0.2669,0.0,12.8120,0.0005,3.6299,-0.0318,3.0667,3.0667,2.1667
3,4,max_k_cut_pairwise_trajectory,12,60,51,51,0.2669,0.0,12.8120,0.0005,3.6299,-0.0318,3.0667,3.0667,6.8333
4,5,max_k_cut_mst_range_guard,12,60,49,49,0.0108,0.0,0.4722,0.0005,3.6298,-0.0318,2.4167,2.4167,0.6667


In [16]:
display(group_variant_summary_df.round(4))


,instance_group,variant_order,variant,source_instances,reoptimization_instances,solved_reoptimization_instances,optimal_reoptimization_instances,mean_gap_to_baseline_percent,median_gap_to_baseline_percent,max_gap_to_baseline_percent,mean_total_variant_seconds,mean_total_pipeline_seconds,mean_time_delta_to_baseline_seconds
0,WORK,1,baseline_all_facilities,4,20,20,20,0.0000,0.0,0.0000,0.0251,3.7833,0.0000
1,WORK,2,max_k_cut_original,4,20,18,18,0.0262,0.0,0.4722,0.0005,3.7588,-0.0245
2,WORK,3,max_k_cut_mst_trajectory,4,20,18,18,0.0262,0.0,0.4722,0.0006,3.7588,-0.0245
3,WORK,4,max_k_cut_pairwise_trajectory,4,20,18,18,0.0262,0.0,0.4722,0.0005,3.7588,-0.0246
4,WORK,5,max_k_cut_mst_range_guard,4,20,18,18,0.0262,0.0,0.4722,0.0005,3.7588,-0.0245
5,NOT WORK,1,baseline_all_facilities,4,20,20,20,0.0000,0.0,0.0000,0.0128,0.4868,0.0000
6,NOT WORK,2,max_k_cut_original,4,20,17,17,0.3238,0.0,3.9916,0.0005,0.4744,-0.0124
7,NOT WORK,3,max_k_cut_mst_trajectory,4,20,17,17,0.7730,0.0,12.8120,0.0004,0.4744,-0.0124
8,NOT WORK,4,max_k_cut_pairwise_trajectory,4,20,17,17,0.7730,0.0,12.8120,0.0004,0.4744,-0.0124
9,NOT WORK,5,max_k_cut_mst_range_guard,4,20,15,15,0.0038,0.0,0.0577,0.0002,0.4742,-0.0126


In [17]:
display(group_summary_wide_df.round(4).T)


,0,1,2
instance_group,WORK,NOT WORK,NOT NECESSARY
source_instances,4,4,4
reoptimization_instances,20,20,20
baseline_all_facilities__mean_gap_to_baseline_percent,0.0,0.0,0.0
max_k_cut_mst_range_guard__mean_gap_to_baseline_percent,0.0262,0.0038,0.0
max_k_cut_mst_trajectory__mean_gap_to_baseline_percent,0.0262,0.773,0.0
max_k_cut_original__mean_gap_to_baseline_percent,0.0262,0.3238,0.0
max_k_cut_pairwise_trajectory__mean_gap_to_baseline_percent,0.0262,0.773,0.0
baseline_all_facilities__mean_total_variant_seconds,0.0251,0.0128,0.059
max_k_cut_mst_range_guard__mean_total_variant_seconds,0.0005,0.0002,0.0007


In [18]:
if SAVE_RESULTS_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(RAW_RESULTS_CSV, index=False)
    structures_df.to_csv(STRUCTURES_CSV, index=False)
    variant_summary_df.to_csv(VARIANT_SUMMARY_CSV, index=False)
    group_variant_summary_df.to_csv(GROUP_VARIANT_SUMMARY_CSV, index=False)
    group_summary_wide_df.to_csv(GROUP_SUMMARY_CSV, index=False)

    print(f"Raw results saved to           : {RAW_RESULTS_CSV}")
    print(f"Structures saved to            : {STRUCTURES_CSV}")
    print(f"Variant summary saved to       : {VARIANT_SUMMARY_CSV}")
    print(f"Group/variant summary saved to : {GROUP_VARIANT_SUMMARY_CSV}")
    print(f"Group wide summary saved to    : {GROUP_SUMMARY_CSV}")

    if not failures_df.empty:
        failures_df.to_csv(FAILURES_CSV, index=False)
        print(f"Failures saved to              : {FAILURES_CSV}")


Raw results saved to           : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_tsplib_group_raw.csv
Structures saved to            : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_tsplib_group_structures.csv
Variant summary saved to       : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_tsplib_group_variant_summary.csv
Group/variant summary saved to : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_tsplib_group_variant_by_group_summary.csv
Group wide summary saved to    : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_tsplib_group_summary_wide.csv
Failures saved to              : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_tsplib_group_failures.csv
